In [ ]:
import torch
import numpy as np
from Network.vae_meassured import VAE20
from torch.utils.data import DataLoader
from Util.data_loader import loader_cre
from Util.metrics import SSIM
from Util.train_41utils import train_vae_41, plot_curve
from Util.Visualize import Plot_2
import os
import torch.optim as optim

In [ ]:
import numpy as np
import random
file_dir = r"..\v41_data\data_noise_0.1.npy"
data_np = np.load(file_dir)
data_np = torch.from_numpy(data_np).float()
#indices_to_remove = np.arange(12, 1000, 11) 
#data_np = np.delete(data_np, indices_to_remove, axis=0)
print(data_np.shape)

In [ ]:
data_input = data_np.unsqueeze(1)

In [ ]:
batch_size = 64
from torch.utils.data import random_split
train_dataset, val_dataset = random_split(
    data_input, 
    [850,151],
    generator=torch.Generator().manual_seed(26) 
)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
#test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
import importlib
import Network.vae_meassured as vae_meassured

importlib.reload(vae_meassured)
model = vae_meassured.VAE20()

#model.load_state_dict(torch.load('./weight/size41_0130test1.pth')) #网络参数初始化
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

LEARNING_RATE = 5e-5
NUM_EPOCHS = 500
#BETA = 0.01 # KL 散度权重
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# if not os.path.exists("./weight"):
#     os.makedirs(".weight")
# ----------------------------------------------------
# 模型的训练函数调用也需要更新，传入 val_loader
# 假设您的 train_vae 函数也返回验证损失历史记录
import importlib
from Util import train_41utils
importlib.reload(train_41utils)
# reload 后重新取函数（确保指向最新版本）
train_vae_41 = train_41utils.train_vae_41
save_path = "./weight/measured_data.pth"
trained_model, losses, recon, kl, val_losses, l3_losses = train_vae_41(
    model = model, 
    train_loader = train_loader, 
    val_loader = val_loader, 
    optimizer = optimizer, 
    num_epochs = NUM_EPOCHS, 
    device = device, 
    beta_decay = True, 
    beta = 2, 
    save_path = save_path,
    L3 =True,
    gamma = 0.5
    )

In [ ]:
plot_curve(losses, recon, kl, val_losses, l3_losses, log_scale=True)

In [ ]:
import matplotlib.pyplot as plt
def l3_per_sample(x_recon, x, target_pos=(10, 10)):
    x_loc, y_loc = target_pos
    # 适配 (B,1,H,W)；如果你是 (B,H,W) 把 [:,0,...] 改成 [:,...]
    delta = (x_recon[:, 0, x_loc, y_loc] - x[:, 0, x_loc, y_loc])  # shape: (B,)
    xnor = x[:, 0, x_loc, y_loc]  # 避免除0
    return delta / xnor  # (B,)

# 随机挑 batch 里的 K 个样本展示
def show_recon_with_l3(model, loader, device, K=5, target_pos=(10,10), vmin=None, vmax=None, cmap='viridis'):
    model.eval()
    with torch.no_grad():
        batch = next(iter(loader))
        # 适配你的 DataLoader 输出：有的返回 (x, y)，有的只返回 x
        x = batch[0] if isinstance(batch, (list, tuple)) else batch
        x = x.to(device)

        # 前向推理（按你 VAE 的 forward 返回来改）
        out = model(x)
        if isinstance(out, (list, tuple)):
            # 常见：x_recon, mu, logvar
            x_recon = out[0]
        else:
            x_recon = out

        l3_vals = l3_per_sample(x_recon, x, target_pos=target_pos).detach().cpu()

        # 从 batch 里挑 K 个索引
        B = x.size(0)
        K = min(K, B)
        idxs = torch.randperm(B)[:K]

        
        for i, idx in enumerate(idxs):
            ori = x[idx, 0].detach().cpu()
            rec = x_recon[idx, 0].detach().cpu()
            l3  = float(l3_vals[idx])

            plt.figure(figsize=(8, 3))
            plt.suptitle(f"sample={int(idx)}   L3@{target_pos}={l3:.6f}")
        
            plt.subplot(1, 2, 1)
            plt.title("Original")
            plt.imshow(ori, cmap=cmap, vmin=vmin, vmax=vmax)
            #plt.scatter([target_pos[1]], [target_pos[0]], s=30)  # 标一下(20,20)
            plt.colorbar()

            plt.subplot(1, 2, 2)
            plt.title("Reconstruction")
            plt.imshow(rec, cmap=cmap, vmin=vmin, vmax=vmax)
            #plt.scatter([target_pos[1]], [target_pos[0]], s=30)
            plt.colorbar()

            plt.tight_layout()
            plt.show()

# 用法：看训练集或验证集随便说明
show_recon_with_l3(trained_model, val_loader, device, K=5, target_pos=(10,10), vmin=0, vmax=1, cmap='viridis')


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1) 取前500条（保持原始顺序）
x500 = data_input[:500]  # (500,1,41,41)

# 2) 转成 torch.Tensor
if isinstance(x500, np.ndarray):
    x500 = torch.from_numpy(x500)
x500 = x500.float()

# 3) 只加载一次模型
model.eval()

# 4) 批量推理拿到 z: (500,8)
bs = 128
Z_list = []
MU_list = []

with torch.no_grad():
    for i in range(0, x500.shape[0], bs):
        xb = x500[i:i+bs].to(device)  # (B,1,41,41)
        # 编码器输出
        mu, logvar = model.encode(xb)
        mu = mu.view(mu.shape[0], -1)   # 保证 (B,8)
        MU_list.append(mu.cpu())
    
MU = torch.cat(MU_list, dim=0).numpy()

x = np.arange(MU.shape[0])  # 0~499

plt.figure(figsize=(12, 10))
x_min, x_max = x.min(), x.max()
y_min, y_max = MU.min(), MU.max()
for k in range(16):
    ax = plt.subplot(8, 2, k + 1)
    ax.plot(x, MU[:, k])
    ax.set_title(f"Latent dim {k+1}")
    ax.set_xlabel("Sample index (0~499)")
    ax.set_ylabel("Value")
    ax.grid(True, alpha=0.3)
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)

plt.tight_layout()
plt.show()

